<a href="https://colab.research.google.com/github/nageshnnazare/pytorch-expts/blob/main/4_datasets_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import make_classification

In [ ]:
# Synthetic classification
X, y = make_classification(
      n_samples=10,
      n_features=2,
      n_informative=2,
      n_redundant=0,
      n_classes=2,
      random_state=42)
print(f"{X.shape}, {y.shape}")

(10, 2), (10,)


In [ ]:
# Convert to torch
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
# Create dataset
dataset = CustomDataset(X, y)
print(f"dataset size = {len(dataset)}")

# Create Dataloader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True) # SequentialSampler / RandomSampler
print(f"dataloader size = {len(dataloader)}")
for bf, bl in dataloader:
  print("-"*80)
  print(f"{bf} : {bl}")

dataset size = 10
dataloader size = 5
--------------------------------------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [-2.8954,  1.9769]]) : tensor([1., 0.])
--------------------------------------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [ 1.7774,  1.5116]]) : tensor([1., 1.])
--------------------------------------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-1.1402, -0.8388]]) : tensor([0., 0.])
--------------------------------------------------------------------------------
tensor([[-0.7206, -0.9606],
        [ 1.8997,  0.8344]]) : tensor([0., 1.])
--------------------------------------------------------------------------------
tensor([[-0.9382, -0.5430],
        [-0.5872, -1.9717]]) : tensor([1., 0.])


# Mini-batch Gradient Descent

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Load dataset
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

# drop id / unnamed col
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

# Preprocessing
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:],
                                                    df.iloc[:, 0],
                                                    test_size=0.2)
# Normalize input
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Relabel output
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Numpy to Pytorch handover
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [ ]:
# Define custom dataset
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
# Create datasets
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

# Create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
# Parameters
learning_rate = 0.1
epochs = 101
epsilon = 1e-6

# Define Model
class NeuralNetwork(nn.Module):
    def __init__(self, num_features):
      super().__init__()
      self.network = nn.Sequential(
          nn.Linear(num_features, 1),
          nn.Sigmoid()
      )
      self.loss_func = nn.BCELoss()
      self.optim = torch.optim.SGD(self.parameters(), lr=learning_rate)

    def forward(self, features):
      out = self.network(features)
      return out

    def loss(self, y_pred, y):
        return self.loss_func(y_pred, y)

In [ ]:
# Create Model
model = NeuralNetwork(X_train_tensor.shape[1])

# Training loop
for epoch in range(epochs):

    for bfeatures, blabels in train_dataloader:
        # Forward pass
        y_pred = model(bfeatures)

        # Loss computation
        loss = model.loss(y_pred, blabels.view(-1, 1))

        # clear grads
        model.optim.zero_grad()

        # Backprop
        loss.backward()

        # Update params
        model.optim.step()
    if epoch % 10 == 0:
        print(f"epoch : {epoch}, loss = {loss}")

epoch : 0, loss = 0.2213025838136673
epoch : 10, loss = 0.0847964882850647
epoch : 20, loss = 0.010206292383372784
epoch : 30, loss = 0.1056678295135498
epoch : 40, loss = 0.04624718800187111
epoch : 50, loss = 0.040227506309747696
epoch : 60, loss = 0.022325783967971802
epoch : 70, loss = 0.004261182155460119
epoch : 80, loss = 0.03363356366753578
epoch : 90, loss = 0.03492853790521622
epoch : 100, loss = 0.04031315818428993


In [ ]:
accurary = []
# Evaluate model
with torch.no_grad():
  for bfeatures, blabels in test_dataloader:
      y_pred = model.forward(bfeatures)
      y_pred_class = (y_pred > 0.7).float()
      acc = (y_pred_class.view(-1) == blabels).float().mean().item()
      accurary.append(acc)

final_accuracy = sum(accurary) / len(accurary)
print(f"accuracy = {final_accuracy*100}%")

accuracy = 59.96154099702835%
